In [2]:
import os
import random
import pandas as pd
import matplotlib.pyplot as plt
import wfdb

# 1. Configuration
# Change this to where your main mimic demo "files" folder sits
MIMIC_DEMO_DIR = "../projectii/patients" 
OUTPUT_DB_DIR = "./data_ingestion/multi_patient_db"
os.makedirs(OUTPUT_DB_DIR, exist_ok=True)

# Pre-defined variations of medical findings to make each patient unique
DISEASES = ["Congestive Heart Failure", "Acute Pneumonia", "Chronic Hypertension", "Myocardial Ischemia"]
XRAY_FINDINGS = [
    "Enlarged cardiac silhouette with mild bilateral pleural effusions.",
    "Patchy airspace opacities in the right lower lobe, consistent with infection.",
    "Clear lung fields. Normal cardiac size. No acute pulmonary disease.",
    "Mild pulmonary vascular congestion with horizontal lines at the lung bases."
]

def scan_and_generate():
    generated_count = 0
    
    # 2. Iterate through the patient folders (e.g., p10001217)
    for p_folder in os.listdir(MIMIC_DEMO_DIR):
        p_path = os.path.join(MIMIC_DEMO_DIR, p_folder)
        if not os.path.isdir(p_path) or not p_folder.startswith('p'):
            continue
            
        # Look inside for the study subfolder (e.g., s102172660)
        s_folders = [f for f in os.listdir(p_path) if f.startswith('s')]
        if not s_folders:
            continue
        s_folder = s_folders[0]
        
        # Get the actual record name inside the study folder
        study_path = os.path.join(p_path, s_folder)
        record_files = [f for f in os.listdir(study_path) if f.endswith('.hea')]
        if not record_files:
            continue
        record_name = record_files[0].replace('.hea', '')
        full_record_path = os.path.join(study_path, record_name)
        
        # Set up destination
        patient_id = p_folder.replace('p', '')
        dest_patient_dir = os.path.join(OUTPUT_DB_DIR, f"patient_{patient_id}")
        os.makedirs(dest_patient_dir, exist_ok=True)
        
        try:
            # 3. Read real ECG Signal
            signals, fields = wfdb.rdsamp(full_record_path)
            num_leads = fields['n_sig']
            
            # Plot ECG and save it
            plt.figure(figsize=(10, 5))
            for i in range(min(3, num_leads)):
                plt.subplot(3, 1, i + 1)
                plt.plot(signals[:2500, i], color='darkblue', linewidth=0.8)
                plt.grid(True, color='pink', linestyle='--', linewidth=0.5)
            plt.tight_layout()
            plt.savefig(os.path.join(dest_patient_dir, "ecg_plot.jpg"), dpi=100)
            plt.close()
            
            # 4. Generate Matched Tabular Data (Labs)
            disease_idx = generated_count % len(DISEASES)
            wbc = random.choice([14.5, 6.2, 11.1, 8.5])
            bnp = random.choice([4500, 120, 2100, 95])
            
            mock_labs = {
                'Test_Name': ['White Blood Cell (WBC)', 'Hemoglobin', 'NT-proBNP', 'Troponin-T'],
                'Result': [wbc, 13.5, bnp, random.choice([0.15, 0.01, 0.03, 0.45])],
                'Unit': ['x10^3/uL', 'g/dL', 'pg/mL', 'ng/mL'],
                'Flag': ['HIGH' if wbc > 11.0 else 'NORMAL', 'NORMAL', 'HIGH' if bnp > 300 else 'NORMAL', 'HIGH']
            }
            pd.DataFrame(mock_labs).to_csv(os.path.join(dest_patient_dir, "lab_results.csv"), index=False)
            
            # 5. Generate Text Admission Note
            note_text = f"""Patient ID: {patient_id}
            Clinical Presentation: Patient admitted presenting signs highly indicative of {DISEASES[disease_idx]}.
            Diagnostic Studies: A 12-lead ECG waveform was captured for study registry {record_name}.
            Imaging Findings: Chest radiograph completed. Analysis indicates: {XRAY_FINDINGS[disease_idx]}
            Labs: Chemistry panels processed and saved to tabular history."""
            
            with open(os.path.join(dest_patient_dir, "admission_note.txt"), "w") as f:
                f.write(note_text)
                
            # 6. Copy over a placeholder chest X-ray
            # (In production, replace this with a random image from an open dataset)
            # For now, we reuse your ECG image or any dummy jpg just to satisfy the vision pipeline
            plt.figure(figsize=(4,4))
            plt.text(0.5, 0.5, f"Chest X-Ray\nPatient: {patient_id}\n{DISEASES[disease_idx]}", ha='center', va='center')
            plt.savefig(os.path.join(dest_patient_dir, "chest_xray.jpg"))
            plt.close()
            
            print(f"Successfully generated database profile for Patient {patient_id}")
            generated_count += 1
            if generated_count >= 10: # Stop after 10 unique patients
                break
                
        except Exception as e:
            print(f"Skipping record {record_name} due to error: {e}")

if __name__ == "__main__":
    scan_and_generate()
    print("\nDatabase generation complete! You now have a 10-patient multimodal data vault.")


FileNotFoundError: [Errno 2] No such file or directory: '../projectii/patients'

In [3]:
import os
from datasets import load_dataset

DB_DIR = "./data_ingestion/multi_patient_db"

patient_disease_map = {
    "10001217": "Pneumonia",         # # 1. Acute Pneumonia
    "10002428": "Infiltration",      # # 2. Myocardial Ischemia 
    "10004733": "No Finding",        # # 3. Chronic Hypertension 
    "10006053": "Infiltration",      # # 4. Myocardial Ischemia 
    "10007058": "Cardiomegaly",      # # 5. Congestive Heart Failure 
    "10008454": "No Finding",        # # 6. Chronic Hypertension 
    "10009628": "Cardiomegaly",      # # 7. Congestive Heart Failure
    "10010867": "Pneumonia",         # # 8. Acute Pneumonia
    "10011398": "Cardiomegaly",      # # 9. Congestive Heart Failure
    "10013049": "Pneumonia"          # # 10. Acute Pneumonia
}

print("Connecting to NIH Medical Stream on Hugging Face...")
# Stream the dataset instantly without downloading the full archive to your disk
dataset = load_dataset("BahaaEldin0/NIH-Chest-Xray-14", split="train", streaming=True)

# Build a search index map for our required categories
needed_findings = set(patient_disease_map.values())
captured_images = {}

print(f"Searching for real chest radiographs matching: {needed_findings}")
for sample in dataset:
    finding = sample["label"]
    
    for target in needed_findings:
        if target in finding and target not in captured_images:
            captured_images[target] = sample["image"]
            print(f" -> Found a valid clinical image for category: [{target}]")
            
    if len(captured_images) == len(needed_findings):
        break

print("\nInjecting realistic clinical images into your multi-patient directory...")

# Iterate through your 10 active patient folders
for folder in os.listdir(DB_DIR):
    if not folder.startswith("patient_"):
        continue
        
    patient_id = folder.replace("patient_", "")
    
    if patient_id in patient_disease_map:
        target_finding = patient_disease_map[patient_id]
        target_folder_path = os.path.join(DB_DIR, folder)
        xray_output_path = os.path.join(target_folder_path, "chest_xray.jpg")
        
        # Get the distinct medical image asset matching this condition
        raw_pil_image = captured_images[target_finding]
        
        # Overwrite the text-box placeholder with real anatomical data
        raw_pil_image.convert("RGB").save(xray_output_path, "JPEG", quality=90)
        print(f" ✅ Overwrote placeholder: {folder}/chest_xray.jpg with real [{target_finding}] asset.")

print("\nSuccess! Your RAG directory contains genuine, clinically distinct vision layers.")


Connecting to NIH Medical Stream on Hugging Face...


Resolving data files:   0%|          | 0/73 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/73 [00:00<?, ?it/s]

Searching for real chest radiographs matching: {'No Finding', 'Pneumonia', 'Infiltration', 'Cardiomegaly'}
 -> Found a valid clinical image for category: [No Finding]
 -> Found a valid clinical image for category: [Pneumonia]
 -> Found a valid clinical image for category: [Infiltration]
 -> Found a valid clinical image for category: [Cardiomegaly]

Injecting realistic clinical images into your multi-patient directory...
 ✅ Overwrote placeholder: patient_10001217/chest_xray.jpg with real [Pneumonia] asset.
 ✅ Overwrote placeholder: patient_10006053/chest_xray.jpg with real [Infiltration] asset.
 ✅ Overwrote placeholder: patient_10010867/chest_xray.jpg with real [Pneumonia] asset.
 ✅ Overwrote placeholder: patient_10004733/chest_xray.jpg with real [No Finding] asset.
 ✅ Overwrote placeholder: patient_10011398/chest_xray.jpg with real [Cardiomegaly] asset.
 ✅ Overwrote placeholder: patient_10007058/chest_xray.jpg with real [Cardiomegaly] asset.
 ✅ Overwrote placeholder: patient_10002428/c

In [4]:
# The project's README outlines a strategy for overcoming healthcare data silos by implementing a specialized, four-modality ingestion pipeline to construct 10 Multimodal Patient Case Profiles. This approach integrates ECG signals from PhysioNet, anatomical imaging from open archives, tabular lab data modeled on MIMIC-IV schema, and generated audio consultations to ensure genuine clinical reasoning. You can integrate this structured data methodology directly into your README.md to demonstrate robust data alignment skills

In [ ]:
!pip install openai-whisper
import whisper

model = whisper.load_model("base")  # small, fast, good enough

def transcribe_audio(audio_path: str) -> str:
    result = model.transcribe(audio_path)
    return result["text"]

# test it
query = transcribe_audio("../projectii/data_ingestion/test_audio/query_1.mp3")
print(f"Transcribed: {query}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 233.6 kB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 581.8 kB/s eta 0:00:00:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 317.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 51.6 kB/s eta 0:00:0000:0100:03m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 MB 3.0 MB/s eta 0:00:0000:0100:010m
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=f7f6c6203da333505803977d67b32b9d78f0801705e7da134b0e8d980b8d4f74
  Stored in directory: /home/naman/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


100%|███████████████████████████████████████| 139M/139M [00:20<00:00, 7.16MiB/s]


Transcribed:  Patient 1001217 came in with chest pain. What are their lab results show and is there anything concerning in the chest x-ray?


In [3]:
import os
import urllib.request

# Automatically build your clean destination folder path
os.makedirs("./data_ingestion/clinical_trials", exist_ok=True)
local_save_path = "./data_ingestion/clinical_trials/heart_failure_guidelines_2022.pdf"

# Clean, lightweight academic direct-download path (Forces instant PDF fetch)
target_url = "https://onlinecjc.ca"

print("📥 Launching download client for the condensed Heart Failure Reference Guide...")

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Accept': 'application/pdf,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'
}

try:
    req = urllib.request.Request(target_url, headers=headers)
    with urllib.request.urlopen(req, timeout=30) as response:
        with open(local_save_path, 'wb') as out_file:
            out_file.write(response.read())
            
    if os.path.getsize(local_save_path) > 5000:
        print(f"✅ Success! Condensed guide saved directly to: {local_save_path}")
        print(f"📦 File Size: {os.path.getsize(local_save_path) / 1024:.1f} KB (Perfect for fast local RAG!)")
    else:
        raise ValueError("Downloaded file is empty or corrupted placeholder text.")
except Exception as network_error:
    print(f"❌ Connection failed: {network_error}")
    print("If your script is blocked, simply paste this unblocked URL into your browser to save it manually:")
    print(target_url)


📥 Launching download client for the condensed Heart Failure Reference Guide...
❌ Connection failed: HTTP Error 403: Forbidden
If your script is blocked, simply paste this unblocked URL into your browser to save it manually:
https://onlinecjc.ca


In [2]:

import chromadb

client = chromadb.PersistentClient(path="./db")
collection = client.get_collection("pharma_guidelines")
data = collection.get(include=["documents"])

lengths = [len(d) for d in data["documents"]]
print(f"Max chunk length:  {max(lengths)} chars")
print(f"Mean chunk length: {sum(lengths)//len(lengths)} chars")
print(f"Chunks over 1500 chars: {sum(1 for l in lengths if l > 1500)}")

Max chunk length:  1939 chars
Mean chunk length: 905 chars
Chunks over 1500 chars: 26


In [1]:
# test each node alone
from langchain_huggingface import HuggingFaceEmbeddings
from src.retrieval.hybrid_retriever import load_vector_stores

embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
vector_stores = load_vector_stores("./db", embedder)

# test classify alone
from src.pipeline.nodes import classify_node
state = {
    "transcribed_query": "What are the lab results?",
    "embedder": embedder,
    "vector_stores": vector_stores
}
result = classify_node(state)
print(result["query_intent"])           # should print "patient_history"
print(result["collections_to_search"])  # should print ["patient_records"]b


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Node 2 - Classify] intent=patient_history, collections=['patient_records']
patient_history
['patient_records']


In [1]:
# test each node alone
from langchain_huggingface import HuggingFaceEmbeddings
from src.retrieval.hybrid_retriever import load_vector_stores

embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
vector_stores = load_vector_stores("./db", embedder)

# test classify alone
from src.pipeline.nodes import retrieve_node
state = {
    "transcribed_query": "What are the lab results?",
    "patient_id" :  "patient_10001217",
    "collections_to_search": ["patient_records"],
    "vector_stores": vector_stores,
}
result = retrieve_node(state)
print(result["retrieved_chunks"])           

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:flashrank.Ranker:Downloading ms-marco-MiniLM-L-12-v2...
ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:02<00:00, 10.8MiB/s]


[Node 3 - Retrieve] Got 3 chunks
[Document(metadata={'id': 3, 'relevance_score': np.float32(0.0376463), 'content_type': 'text', 'source': 'data_ingestion/multi_patient_db/patient_10001217/patient_report.pdf', 'element_index': 1, 'file_name': 'patient_report.pdf', 'section_headings': 'I. Clinical Presentation & Case Narrative', 'patient_id': 'patient_10001217', 'page_number': 1}, page_content='Patient ID: 10001217\nClinical Presentation: Patient admitted presenting signs highly indicative of Acute Pneumonia.\nDiagnostic Studies: A 12-lead ECG waveform was captured for study registry 102172660.\nImaging Findings: Chest radiograph completed. Analysis indicates: Patchy airspace opacities in the right lower lobe, consistent with infection.\nLabs: Chemistry panels processed and saved to tabular history.'), Document(metadata={'id': 0, 'relevance_score': np.float32(0.026686601), 'patient_id': 'patient_10001217', 'content_type': 'medical_image', 'image_path': 'extracted_images/patient_10001217/

In [2]:
from src.pipeline.nodes import generate_node
state = {
    "transcribed_query": "What are the lab results?",
    "retrieved_chunks" : result["retrieved_chunks"]
}
result_2 = generate_node(state)
print(result_2["answer"], result_2["sources"])


FULL CONTEXT SENT TO MODEL:
Source 1 (text from patient_report.pdf page 1):
Patient ID: 10001217
Clinical Presentation: Patient admitted presenting signs highly indicative of Acute Pneumonia.
Diagnostic Studies: A 12-lead ECG waveform was captured for study registry 102172660.
Imaging Findings: Chest radiograph completed. Analysis indicates: Patchy airspace opacities in the right lower lobe, consistent with infection.
Labs: Chemistry panels processed and saved to tabular history.

---

Source 2 (medical_image from patient_report.pdf page 2):
Medical Image — Page 2:
 The image you've provided appears to be a screenshot of an ECG (Electrocardiogram) waveform with accompanying data. This type of imaging is used in cardiology to assess the electrical activity of the heart. Here are the visible findings:

1. **ECG Waveform**: The central part of the image shows a series of waves, each representing a different phase of the heart's electrical cycle. These waves are labeled with numbers that 

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


[Node 4 - Generate] 184 chars
 The lab results are as follows:
- White Blood Cell (WBC): 8.5 x10^3/uL (NORMAL)
- Hemoglobin: 13.5 g/dL (NORMAL)
- NT-proBNP: 120.0 pg/mL (NORMAL)
- Troponin-T: 0.15 ng/mL ⚠️ CRITICAL [{'index': 1, 'file': 'patient_report.pdf', 'page': 1, 'type': 'text', 'preview': 'Patient ID: 10001217\nClinical Presentation: Patient admitted presenting signs highly indicative of A'}, {'index': 2, 'file': 'patient_report.pdf', 'page': 2, 'type': 'medical_image', 'preview': "Medical Image — Page 2:\n The image you've provided appears to be a screenshot of an ECG (Electrocard"}, {'index': 3, 'file': 'patient_report.pdf', 'page': 1, 'type': 'table', 'preview': 'White Blood Cell (WBC), Result = 8.5. White Blood Cell (WBC), Unit = x10^3/uL. White Blood Cell (WBC'}]


In [4]:
from src.pipeline.nodes import verify_node
state = {
    "answer" : result_2["answer"],
    "retry_count" : 0
}
resultt = verify_node(state)
print(resultt["is_grounded"], resultt["retry_count"])

[Node 5 - Verify] ⚠️  Potentially fabricated values: {'0.15', '120.0', '8.5', '13.5'}
[Node 5 - Verify] grounded=False, retries=0
False 1


In [3]:
import chromadb

client = chromadb.PersistentClient(path="./db")
collection = client.get_collection("patient_records")

results = collection.get(
    where={"patient_id": "patient_10001217"},
    include=["documents", "metadatas"]
)

for doc, meta in zip(results["documents"], results["metadatas"]):
    print(f"Type: {meta.get('content_type')}")
    print(f"Preview: {doc[:150]}")
    print("---")

Type: text
Preview: Patient Identity System ID:
#10001217
---
Type: text
Preview: Patient ID: 10001217
Clinical Presentation: Patient admitted presenting signs highly indicative of Acute Pneumonia.
Diagnostic Studies: A 12-lead ECG 
---
Type: table
Preview: White Blood Cell (WBC), Result = 8.5. White Blood Cell (WBC), Unit = x10^3/uL. White Blood Cell (WBC), Flag = NORMAL. Hemoglobin, Result = 13.5. Hemog
---
Type: text
Preview: Asset Record Profile:
chest_xray.jpg
Asset Record Profile: ecg_plot.jpg
---
Type: medical_image
Preview: Medical Image — Page 2:
 The image provided appears to be a chest X-ray of a patient with an acute pneumonia diagnosis. Here's a clinical description 
---
Type: medical_image
Preview: Medical Image — Page 2:
 The image you've provided appears to be a screenshot of an ECG (Electrocardiogram) waveform with accompanying data. This type
---
